# GV-YOLO26 Training on Kaggle

Trains **GV-YOLO26** (GhostConv + VoVGSCSPC backbone/neck, LGDH head, IoU+NWD box loss) on a free Kaggle GPU, reproducing the 6-stage ablation ladder from `train.py`.

**Before running, set in Notebook Settings (right sidebar):**
1. **Accelerator -> GPU** (T4 x2 or P100).
2. **Internet -> On** (needed for `pip install` and to auto-download the YOLO26 COCO checkpoint).
3. If you want this notebook to pull the dataset straight from Roboflow, add a **Secret** named `ROBOFLOW_API_KEY` (Add-ons -> Secrets, then attach it to this notebook). Otherwise attach the dataset as a **Kaggle Dataset** input (a YOLOv8-format export with `train/`, `valid/`, `test/` folders) and set `DATA_SOURCE = "kaggle_dataset"` in the dataset cell below.

This notebook writes out the full GV-YOLO26 source (the `gv_modules` package, model configs, `train.py`, `val.py`) into `/kaggle/working/GV-YOLO26`, so it runs standalone -- nothing else needs to be uploaded.

For a long run (the full 6-stage ladder easily exceeds a few hours), use **Save & Run All (Commit)** rather than running interactively, so the job keeps going if your browser disconnects.


In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "- cuda available:", torch.cuda.is_available())


## 1. Install dependencies

`ultralytics==8.4.156` is pinned **exactly**: GV-YOLO26 registers its custom layers (`GSConv`, `VoVGSCSPC`, `LGDH`) by patching the source of Ultralytics' `parse_model()` at import time, matched against this exact version's source text. A different version will fail loudly on import rather than silently doing the wrong thing.


In [ ]:
%pip install -q "ultralytics==8.4.156" "albumentations>=2.0" opencv-python-headless


## 2. Write the GV-YOLO26 project files

Reconstructs the project exactly as it exists in the original repo.

In [ ]:
import os

PROJECT_DIR = "/kaggle/working/GV-YOLO26"
for d in ["gv_modules", "models/ablation", "data"]:
    os.makedirs(f"{PROJECT_DIR}/{d}", exist_ok=True)
%cd {PROJECT_DIR}


In [ ]:
%%writefile gv_modules/__init__.py
"""GV-YOLO26 layers, losses and trainers. Importing this package registers the layers with Ultralytics."""

from .register import register

register()

from .blocks import GSBottleneckC, GSConv, VoVGSCSPC  # noqa: E402
from .head import LGDH  # noqa: E402
from .loss import NWDDetectionLoss, nwd  # noqa: E402
from .model import GVDetectionModel  # noqa: E402
from .trainer import DistillTrainer, GVTrainer, NWDTrainer  # noqa: E402

__all__ = (
    "GSConv",
    "GSBottleneckC",
    "VoVGSCSPC",
    "LGDH",
    "nwd",
    "NWDDetectionLoss",
    "GVDetectionModel",
    "GVTrainer",
    "NWDTrainer",
    "DistillTrainer",
)


In [ ]:
%%writefile gv_modules/register.py
"""Make the GV-YOLO26 layers usable from Ultralytics YAML model files.

Ultralytics keeps the lists of known layer types inside `parse_model()`, so they cannot be extended from outside. This
module rewrites that function once at import time to add our layers to the right lists, and fails loudly if the
installed Ultralytics version is laid out differently from the one this project was written for.
"""

import inspect

import ultralytics
import ultralytics.nn.tasks as tasks

from .blocks import GSConv, VoVGSCSPC
from .head import LGDH

TESTED_VERSION = "8.4.156"


def _patch_source(src):
    # base_modules: (c1, c2, ...) with c2 scaled by width; repeat_modules: depth-scaled n inserted as 3rd arg;
    # detect heads: reg_max, end2end and input channels appended
    base_end = "            A2C2f,\n        }\n    )\n    repeat_modules"
    repeat_end = "            A2C2f,\n        }\n    )\n    for i, (f, n, m, args)"
    detect_end = "                OBB26,\n            }\n        ):\n            args.extend([reg_max, end2end"
    for anchor in (base_end, repeat_end, detect_end):
        if src.count(anchor) != 1:
            raise RuntimeError(
                f"gv_modules: Ultralytics {ultralytics.__version__} is not supported "
                f"(parse_model layout changed; tested with {TESTED_VERSION})."
            )
    src = src.replace(base_end, base_end.replace("A2C2f,\n", "A2C2f,\n            GSConv,\n            VoVGSCSPC,\n"))
    src = src.replace(repeat_end, repeat_end.replace("A2C2f,\n", "A2C2f,\n            VoVGSCSPC,\n"))
    src = src.replace(detect_end, detect_end.replace("OBB26,\n", "OBB26,\n                LGDH,\n"))
    return src


def register():
    if getattr(tasks, "_gv_registered", False):
        return
    for name, obj in {"GSConv": GSConv, "VoVGSCSPC": VoVGSCSPC, "LGDH": LGDH}.items():
        setattr(tasks, name, obj)  # parse_model resolves layer names through the tasks module globals
    src = _patch_source(inspect.getsource(tasks.parse_model))
    exec(compile(src, tasks.__file__, "exec"), tasks.__dict__)
    tasks._gv_registered = True


In [ ]:
%%writefile gv_modules/blocks.py
"""Cheap convolution blocks used in the GV-YOLO26 backbone and neck."""

import torch
import torch.nn as nn

from ultralytics.nn.modules.conv import Conv, DWConv


class GSConv(nn.Module):
    """Dense half from a standard convolution, cheap half from a depthwise convolution, then a channel shuffle."""

    def __init__(self, c1, c2, k=1, s=1, act=True):
        super().__init__()
        c_ = c2 // 2
        self.cv1 = Conv(c1, c_, k, s, act=act)  # dense half
        self.cv2 = Conv(c_, c_, 5, 1, None, c_, act=act)  # depthwise half

    def forward(self, x):
        x1 = self.cv1(x)
        x2 = torch.cat((x1, self.cv2(x1)), 1)
        b, n, h, w = x2.shape  # channel shuffle: interleave the dense and cheap halves
        return x2.view(b, 2, n // 2, h, w).transpose(1, 2).reshape(b, n, h, w)


class GSBottleneckC(nn.Module):
    """Two GSConv layers with a cheap depthwise shortcut."""

    def __init__(self, c1, c2, k=1, s=1):
        super().__init__()
        c_ = c2 // 2
        self.conv = nn.Sequential(GSConv(c1, c_, 1, 1), GSConv(c_, c2, 3, 1, act=False))
        self.shortcut = DWConv(c1, c2, k, s, act=False)

    def forward(self, x):
        return self.conv(x) + self.shortcut(x)


class VoVGSCSPC(nn.Module):
    """Cross-stage partial block with GSConv bottlenecks and one-shot aggregation."""

    def __init__(self, c1, c2, n=1, e=0.5):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        self.gsb = nn.Sequential(*(GSBottleneckC(c_, c_) for _ in range(n)))
        self.cv3 = Conv(2 * c_, c2, 1)

    def forward(self, x):
        return self.cv3(torch.cat((self.cv2(x), self.gsb(self.cv1(x))), 1))


In [ ]:
%%writefile gv_modules/head.py
"""Lightweight Grouped Detection Head (LGDH) on top of the YOLO26 end-to-end Detect head."""

import copy

import torch
import torch.nn as nn

from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.modules.head import Detect


class LGDH(Detect):
    """Detect head whose box and class outputs share one grouped 3x3 stem per level.

    The one-to-many branch (training) and the one-to-one branch (NMS-free inference) each get their own copy of the
    stem and output convolutions, exactly as YOLO26 duplicates its default head.
    """

    channels_per_group = 8  # groups = c / 8 in each 3x3 stem convolution

    def __init__(self, nc=80, reg_max=16, end2end=False, ch=()):
        super().__init__(nc, reg_max, end2end, ch)
        g = lambda c: max(c // self.channels_per_group, 1)  # noqa: E731
        self.stem = nn.ModuleList(nn.Sequential(Conv(x, x, 3, g=g(x)), Conv(x, x, 3, g=g(x))) for x in ch)
        # single-layer Sequentials keep Detect.bias_init() (which touches [-1]) working unchanged
        self.cv2 = nn.ModuleList(nn.Sequential(nn.Conv2d(x, 4 * self.reg_max, 1)) for x in ch)
        self.cv3 = nn.ModuleList(nn.Sequential(nn.Conv2d(x, self.nc, 1)) for x in ch)
        if end2end:
            self.one2one_stem = copy.deepcopy(self.stem)
            self.one2one_cv2 = copy.deepcopy(self.cv2)
            self.one2one_cv3 = copy.deepcopy(self.cv3)

    @property
    def one2many(self):
        return {"box_head": self.cv2, "cls_head": self.cv3, "stem": self.stem}

    @property
    def one2one(self):
        return {"box_head": self.one2one_cv2, "cls_head": self.one2one_cv3, "stem": self.one2one_stem}

    def forward_head(self, x, box_head=None, cls_head=None, stem=None):
        if box_head is None or cls_head is None:  # branch removed by fuse()
            return {}
        bs = x[0].shape[0]
        f = [stem[i](x[i]) for i in range(self.nl)]  # shared stem, computed once per level
        boxes = torch.cat([box_head[i](f[i]).view(bs, 4 * self.reg_max, -1) for i in range(self.nl)], dim=-1)
        scores = torch.cat([cls_head[i](f[i]).view(bs, self.nc, -1) for i in range(self.nl)], dim=-1)
        return {"boxes": boxes, "scores": scores, "feats": x}


In [ ]:
%%writefile gv_modules/loss.py
"""NWD box loss and BCKD + CWD distillation losses for GV-YOLO26."""

import torch
import torch.nn.functional as F

from ultralytics.utils.loss import BboxLoss, v8DetectionLoss
from ultralytics.utils.metrics import bbox_iou
from ultralytics.utils.tal import dist2bbox, make_anchors


def nwd(pred, target, C=12.8, eps=1e-7):
    """Normalized Gaussian Wasserstein Distance between xyxy boxes given in pixels (equations 4.2 and 4.3)."""
    pc = (pred[:, :2] + pred[:, 2:]) / 2
    tc = (target[:, :2] + target[:, 2:]) / 2
    pwh = (pred[:, 2:] - pred[:, :2]).clamp(min=eps)
    twh = (target[:, 2:] - target[:, :2]).clamp(min=eps)
    d_center = ((pc - tc) ** 2).sum(dim=1)
    d_size = (((pwh - twh) / 2) ** 2).sum(dim=1)
    w2 = d_center + d_size + eps  # squared Wasserstein-2 distance
    return torch.exp(-torch.sqrt(w2) / C)


class NWDBboxLoss(BboxLoss):
    """YOLO26 box loss with the IoU term blended with NWD: L_box = alpha * L_IoU + (1 - alpha) * (1 - NWD)."""

    def __init__(self, reg_max=16, alpha=0.5):
        super().__init__(reg_max)
        self.alpha = alpha

    def forward(self, pred_dist, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum,
                fg_mask, imgsz, stride):
        loss_iou, loss_l1 = super().forward(pred_dist, pred_bboxes, anchor_points, target_bboxes, target_scores,
                                            target_scores_sum, fg_mask, imgsz, stride)
        b, a = fg_mask.nonzero(as_tuple=True)
        weight = target_scores[b, a].sum(-1)
        s = stride[a]  # boxes arrive in grid units; NWD's constant C is defined in pixels
        sim = nwd(pred_bboxes[b, a].float() * s, target_bboxes[b, a].float() * s)
        loss_nwd = ((1.0 - sim) * weight).sum() / target_scores_sum
        return self.alpha * loss_iou + (1 - self.alpha) * loss_nwd, loss_l1


class NWDDetectionLoss(v8DetectionLoss):
    """v8DetectionLoss whose box term uses NWDBboxLoss."""

    alpha = 0.5

    def __init__(self, model, tal_topk=10, tal_topk2=None):
        super().__init__(model, tal_topk, tal_topk2)
        self.bbox_loss = NWDBboxLoss(self.reg_max, self.alpha).to(self.device)


def bckd_loss(s, t, stride):
    """Classification and localisation distillation on the head outputs (BCKD style).

    s, t: one-to-many prediction dicts of the student and teacher ({"boxes", "scores", "feats"}).
    """
    t_prob = t["scores"].float().sigmoid()
    s_logit = s["scores"].float()
    w = (t_prob - s_logit.sigmoid()).abs().pow(2)  # focus on locations where the two disagree
    cls = (F.binary_cross_entropy_with_logits(s_logit, t_prob, reduction="none") * w).sum()
    cls = cls / t_prob.sum().clamp(min=1)

    anchors, _ = make_anchors(s["feats"], stride, 0.5)
    sb = dist2bbox(s["boxes"].float().permute(0, 2, 1), anchors, xywh=False)
    tb = dist2bbox(t["boxes"].float().permute(0, 2, 1), anchors, xywh=False)
    wt = t_prob.max(1)[0]  # teacher confidence per location
    giou = bbox_iou(sb, tb, xywh=False, GIoU=True).squeeze(-1)
    box = ((1.0 - giou) * wt).sum() / wt.sum().clamp(min=1)
    return cls + box


def cwd_loss(s_feats, t_feats, adapters, tau=1.0):
    """Channel-wise distillation: per-channel spatial softmax, KL(teacher || student)."""
    total = 0.0
    for sf, tf, ad in zip(s_feats, t_feats, adapters):
        sf = ad(sf).float()
        tf = tf.float()
        n, c, h, w = sf.shape
        log_ps = F.log_softmax(sf.reshape(n * c, h * w) / tau, dim=1)
        pt = F.softmax(tf.reshape(n * c, h * w) / tau, dim=1)
        total = total + (pt * (torch.log(pt + 1e-12) - log_ps)).sum() * tau**2 / (n * c)
    return total / len(s_feats)


In [ ]:
%%writefile gv_modules/model.py
"""GV-YOLO26 detection model: Ultralytics DetectionModel with the NWD criterion and an optional distillation term."""

import torch

from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.loss import E2ELoss, v8DetectionLoss

from .loss import NWDDetectionLoss, bckd_loss, cwd_loss

# The teacher lives here rather than on the model, so it is never deep-copied into the EMA or pickled into checkpoints.
_KD = {"teacher": None, "weight_bckd": 0.5, "weight_cwd": 0.5}


class GVDetectionModel(DetectionModel):
    use_nwd = False

    def init_criterion(self):
        loss_fn = NWDDetectionLoss if self.use_nwd else v8DetectionLoss
        end2end = getattr(self.model[-1], "one2one_cv2", None) is not None
        return E2ELoss(self, loss_fn) if end2end else loss_fn(self)

    def loss(self, batch, preds=None):
        if getattr(self, "criterion", None) is None:
            self.criterion = self.init_criterion()
        if preds is None:
            preds = self.forward(batch["img"])
        loss, items = self.criterion(preds, batch)

        teacher = _KD["teacher"]
        if teacher is None:
            return loss, items
        kd = torch.zeros((), device=loss.device)
        if self.training and hasattr(self, "kd_adapters"):
            s = preds[1] if isinstance(preds, tuple) else preds
            s = s["one2many"] if "one2many" in s else s
            with torch.no_grad():
                t = teacher(batch["img"])[1]  # teacher runs in eval mode: (y, preds)
                t = t["one2many"] if "one2many" in t else t
            kd = _KD["weight_bckd"] * bckd_loss(s, t, self.stride) + _KD["weight_cwd"] * cwd_loss(
                s["feats"], t["feats"], self.kd_adapters
            )
            loss = torch.cat([loss, (kd * batch["img"].shape[0]).view(1).to(loss.dtype)])
        items = {**items, "kd_loss": kd.detach()}  # same keys in training and validation
        return loss, items


In [ ]:
%%writefile gv_modules/trainer.py
"""Trainers for GV-YOLO26: plain, NWD-loss and distillation."""

import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import load_checkpoint
from ultralytics.utils import LOGGER, RANK

from .model import _KD, GVDetectionModel


class GVTrainer(DetectionTrainer):
    """Builds a GVDetectionModel with the standard YOLO26 loss (ablation stages 1-5)."""

    use_nwd = False

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = GVDetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=verbose and RANK == -1)
        model.use_nwd = self.use_nwd
        model = self.set_model_names_for_load(model)
        if weights:
            model.load(weights)
        return model


class NWDTrainer(GVTrainer):
    """GVTrainer with the IoU + NWD box loss (ablation stage 6, the final GV-YOLO26)."""

    use_nwd = True


class DistillTrainer(NWDTrainer):
    """NWDTrainer that adds BCKD + CWD distillation from a frozen teacher. Set `teacher` before training."""

    teacher = None  # path to the trained teacher checkpoint (e.g. GV-YOLO26 at S scale)

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        if not self.teacher:
            raise ValueError("DistillTrainer.teacher must point to a trained teacher checkpoint")
        t, _ = load_checkpoint(self.teacher, device="cpu", fuse=False)
        self._teacher = t.float().eval()
        for p in self._teacher.parameters():
            p.requires_grad_(False)
        s_ch = [b[0].conv.in_channels for b in model.model[-1].stem]
        t_ch = [b[0].conv.in_channels for b in self._teacher.model[-1].stem]
        # 1x1 adapters map student neck features to teacher channels for CWD; trained with the student, never saved
        model.kd_adapters = nn.ModuleList(nn.Conv2d(cs, ct, 1) for cs, ct in zip(s_ch, t_ch))
        return model

    def _setup_train(self):
        super()._setup_train()
        _KD["teacher"] = self._teacher.to(self.device)
        LOGGER.info(f"Distilling from {self.teacher}")

    def save_model(self):
        ema = self.ema.ema
        adapters = ema._modules.pop("kd_adapters", None)  # keep the adapters out of the checkpoint
        try:
            super().save_model()
        finally:
            if adapters is not None:
                ema.kd_adapters = adapters

    def final_eval(self):
        _KD["teacher"] = None
        return super().final_eval()


In [ ]:
%%writefile gv_modules/augment.py
"""Weather and photometric augmentation for GV-YOLO26 (applied on top of the standard Ultralytics recipe)."""


def weather_augmentations():
    import albumentations as A

    return [
        A.RandomRain(drop_length=15, blur_value=3, brightness_coefficient=0.85, p=0.10),
        A.RandomSnow(snow_point_range=(0.1, 0.25), brightness_coeff=1.8, p=0.05),
        A.MotionBlur(blur_limit=(3, 9), p=0.10),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.0, p=0.10),
    ]


In [ ]:
%%writefile models/gv-yolo26.yaml
# GV-YOLO26: YOLO26 with a GhostConv-VoVGSCSPC backbone/neck, P2 level and LGDH head (stage 5/6)
# Use as models/gv-yolo26n.yaml (student) or models/gv-yolo26s.yaml (distillation teacher); the letter selects the scale.

nc: 2 # speed bump, pothole
end2end: True # one-to-one head, NMS-free inference
reg_max: 1 # no DFL
scales: # [depth, width, max_channels]
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]] # 2
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]] # 4
  - [-1, 1, GhostConv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, VoVGSCSPC, [512]] # 6
  - [-1, 1, GhostConv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, VoVGSCSPC, [1024]] # 8
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

head:
  # top-down path, extended to P2
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]] # cat backbone P2
  - [-1, 2, C3k2, [128, True]] # 19 (P2/4-xsmall)

  # bottom-up path
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 16], 1, Concat, [1]] # cat head P3
  - [-1, 2, C3k2, [256, True]] # 22 (P3/8-small)
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, VoVGSCSPC, [512]] # 25 (P4/16-medium)
  - [-1, 1, GhostConv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, VoVGSCSPC, [1024]] # 28 (P5/32-large)

  - [[19, 22, 25, 28], 1, LGDH, [nc]] # P2, P3, P4, P5


In [ ]:
%%writefile models/ablation/gv-yolo26-p2.yaml
# Ablation stage 2: YOLO26 + P2 detection level (stock Detect head)
# Use as models/ablation/gv-yolo26n-p2.yaml (the "n" selects the scale).

nc: 2 # speed bump, pothole
end2end: True # one-to-one head, NMS-free inference
reg_max: 1 # no DFL
scales: # [depth, width, max_channels]
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]] # 2
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]] # 4
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]] # 6
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]] # 8
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

head:
  # top-down path, extended to P2
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]] # cat backbone P2
  - [-1, 2, C3k2, [128, True]] # 19 (P2/4-xsmall)

  # bottom-up path
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 16], 1, Concat, [1]] # cat head P3
  - [-1, 2, C3k2, [256, True]] # 22 (P3/8-small)
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, True]] # 25 (P4/16-medium)
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 28 (P5/32-large)

  - [[19, 22, 25, 28], 1, Detect, [nc]] # P2, P3, P4, P5


In [ ]:
%%writefile models/ablation/gv-yolo26-p2-lgdh.yaml
# Ablation stage 3: + LGDH head
# Use as models/ablation/gv-yolo26n-p2-lgdh.yaml (the "n" selects the scale).

nc: 2 # speed bump, pothole
end2end: True # one-to-one head, NMS-free inference
reg_max: 1 # no DFL
scales: # [depth, width, max_channels]
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]] # 2
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]] # 4
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]] # 6
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]] # 8
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

head:
  # top-down path, extended to P2
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]] # cat backbone P2
  - [-1, 2, C3k2, [128, True]] # 19 (P2/4-xsmall)

  # bottom-up path
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 16], 1, Concat, [1]] # cat head P3
  - [-1, 2, C3k2, [256, True]] # 22 (P3/8-small)
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, True]] # 25 (P4/16-medium)
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 28 (P5/32-large)

  - [[19, 22, 25, 28], 1, LGDH, [nc]] # P2, P3, P4, P5


In [ ]:
%%writefile models/ablation/gv-yolo26-p2-lgdh-ghost.yaml
# Ablation stage 4: + GhostConv downsampling
# Use as models/ablation/gv-yolo26n-p2-lgdh-ghost.yaml (the "n" selects the scale).

nc: 2 # speed bump, pothole
end2end: True # one-to-one head, NMS-free inference
reg_max: 1 # no DFL
scales: # [depth, width, max_channels]
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]] # 2
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]] # 4
  - [-1, 1, GhostConv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]] # 6
  - [-1, 1, GhostConv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]] # 8
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

head:
  # top-down path, extended to P2
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]] # cat backbone P2
  - [-1, 2, C3k2, [128, True]] # 19 (P2/4-xsmall)

  # bottom-up path
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 16], 1, Concat, [1]] # cat head P3
  - [-1, 2, C3k2, [256, True]] # 22 (P3/8-small)
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, True]] # 25 (P4/16-medium)
  - [-1, 1, GhostConv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 28 (P5/32-large)

  - [[19, 22, 25, 28], 1, LGDH, [nc]] # P2, P3, P4, P5


In [ ]:
%%writefile train.py
"""Train one GV-YOLO26 design stage (Table 4.5) with the configuration of Table 4.6.

Examples:
    python train.py --cfg yolo26n.yaml --name s1_yolo26n                                  # stage 1
    python train.py --cfg models/ablation/gv-yolo26n-p2.yaml --name s2_p2                 # stage 2
    python train.py --cfg models/ablation/gv-yolo26n-p2-lgdh.yaml --name s3_lgdh          # stage 3
    python train.py --cfg models/ablation/gv-yolo26n-p2-lgdh-ghost.yaml --name s4_ghost   # stage 4
    python train.py --cfg models/gv-yolo26n.yaml --name s5_vov                            # stage 5
    python train.py --cfg models/gv-yolo26n.yaml --nwd --name s6_gv_yolo26                # stage 6 (final)
    python train.py --cfg models/gv-yolo26s.yaml --nwd --name teacher_s --weights yolo26s.pt   # teacher
"""

import argparse

import gv_modules  # noqa: F401  registers the GV-YOLO26 layers
from gv_modules import GVTrainer, NWDDetectionLoss, NWDTrainer
from gv_modules.augment import weather_augmentations
from ultralytics import YOLO


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--cfg", default="models/gv-yolo26n.yaml", help="model YAML (scale letter after 'yolo26')")
    p.add_argument("--data", default="data/data.yaml")
    p.add_argument("--weights", default="yolo26n.pt", help="COCO checkpoint for layers whose shapes match; '' = none")
    p.add_argument("--nwd", action="store_true", help="use the IoU + NWD box loss (stage 6)")
    p.add_argument("--alpha", type=float, default=0.5, help="IoU weight in the box loss (equation 4.4)")
    p.add_argument("--no-weather", action="store_true", help="disable rain/snow/blur/colour augmentation")
    p.add_argument("--epochs", type=int, default=150)
    p.add_argument("--patience", type=int, default=30)
    p.add_argument("--batch", type=int, default=16)
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--device", default=None)
    p.add_argument("--workers", type=int, default=8)
    p.add_argument("--project", default="runs")
    p.add_argument("--name", default="gv_yolo26")
    return p.parse_args()


def main(opt):
    NWDDetectionLoss.alpha = opt.alpha
    model = YOLO(opt.cfg, task="detect")
    if opt.weights:
        model.load(opt.weights)  # transfers only the layers whose names and shapes match
    extra = {} if opt.no_weather else {"augmentations": weather_augmentations()}
    model.train(
        trainer=NWDTrainer if opt.nwd else GVTrainer,
        data=opt.data,
        imgsz=opt.imgsz,
        epochs=opt.epochs,
        patience=opt.patience,
        batch=opt.batch,
        seed=42,
        deterministic=True,
        optimizer="auto",
        amp=True,
        close_mosaic=10,
        flipud=0.0,
        device=opt.device,
        workers=opt.workers,
        project=opt.project,
        name=opt.name,
        **extra,
    )


if __name__ == "__main__":
    main(parse_args())


In [ ]:
%%writefile val.py
"""Evaluate a trained model once on the test split and time single-image inference (Tables 5.1 and 5.3).

Example:
    python val.py --weights runs/s6_gv_yolo26/weights/best.pt
"""

import argparse
import time
from pathlib import Path

import numpy as np
import torch

import gv_modules  # noqa: F401  registers the GV-YOLO26 layers
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--weights", required=True)
    p.add_argument("--data", default="data/data.yaml")
    p.add_argument("--split", default="test")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--device", default=None)
    p.add_argument("--timing-images", type=int, default=100)
    p.add_argument("--nms", action="store_true", help="use the one-to-many head + NMS instead of the NMS-free head")
    return p.parse_args()


def main(opt):
    model = YOLO(opt.weights)
    nms = None if opt.nms else False  # False selects the one-to-one (NMS-free) head of end-to-end models
    m = model.val(data=opt.data, split=opt.split, imgsz=opt.imgsz, batch=1, conf=0.001, iou=0.7,
                  device=opt.device, plots=True, nms=nms)
    print(f"P {m.box.mp:.3f}  R {m.box.mr:.3f}  mAP@0.5 {m.box.map50:.3f}  mAP@0.5:0.95 {m.box.map:.3f}")
    for i, name in m.names.items():
        print(f"  {name:12s} AP@0.5 {m.box.ap50[i]:.3f}  AP@0.5:0.95 {m.box.ap[i]:.3f}")

    # end-to-end single-image latency: up to 10 warm-up images, then the mean over the next N
    root = Path(check_det_dataset(opt.data)[opt.split])
    files = sorted(str(f) for f in root.rglob("*") if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
    warm = min(10, len(files) // 2)
    files = files[: opt.timing_images + warm]
    times = []
    for k, f in enumerate(files):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model.predict(f, imgsz=opt.imgsz, device=opt.device, nms=nms, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        if k >= warm:
            times.append(time.perf_counter() - t0)
    ms = 1000 * float(np.mean(times))
    print(f"latency {ms:.1f} ms  ->  {1000 / ms:.1f} FPS over {len(times)} images")


if __name__ == "__main__":
    main(parse_args())


## 3. Smoke-test the model build

Builds the final-stage model on whatever device is available, before spending GPU time on a dataset download. This should print a parameter count with no errors.


In [ ]:
import gv_modules  # noqa: F401  registers the GV-YOLO26 layers
from ultralytics import YOLO

m = YOLO("models/gv-yolo26.yaml", task="detect")
print("GV-YOLO26n builds OK -", sum(p.numel() for p in m.model.parameters()), "params")


## 4. Get the dataset

Set `DATA_SOURCE`:
- `"kaggle_dataset"` (default here) - uses the attached Kaggle Dataset [elangovan001/speed12](https://www.kaggle.com/datasets/elangovan001/speed12), mounted somewhere under `/kaggle/input/` (depth varies, auto-detected below).
- `"roboflow"` - downloads [Humps/Bumps & Potholes Detection, v8](https://universe.roboflow.com/detection-system/humps-bumps-potholes-detection/dataset/8) directly from Roboflow using your `ROBOFLOW_API_KEY` secret instead; that export ships its own `data.yaml` with a proper `train/valid/test` split.

**`speed12` ships `train/` (5,725 images, 4,025 with labels - the rest are unlabelled backgrounds) and `test/` (1,125 images, all labelled - this is the original Roboflow v8 test partition, confirmed by file count: it matches exactly what `val.py` reported during evaluation). There is no `valid/` split and no `data.yaml`.** Class order was confirmed by inspecting label files directly (not assumed): class `0` has many small boxes per frame (potholes), class `1` has wide low boxes (speed bumps) - matching the original Roboflow project's `0: Pothole, 1: Speed Bump`.

**Split strategy:** `test/` is left completely untouched - it's never read during training, only in step 7's final evaluation, so those numbers are genuinely blind. A validation split is instead carved out of `train/` (10%, seeded shuffle) purely for training-time checkpoint selection and early stopping. `train/` and `test/` are never recombined and reshuffled together, because `train/`'s "G00xxxxx"-style filenames are sequential dashcam frames - randomly mixing adjacent frames across train/test would leak near-duplicates into the "held-out" set and inflate the reported metrics.


In [ ]:
from pathlib import Path
import random

import yaml

DATA_SOURCE = "kaggle_dataset"  # "roboflow" or "kaggle_dataset"

if DATA_SOURCE == "roboflow":
    %pip install -q roboflow
    from kaggle_secrets import UserSecretsClient
    from roboflow import Roboflow

    api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    rf = Roboflow(api_key=api_key)
    project = rf.workspace("detection-system").project("humps-bumps-potholes-detection")
    dataset = project.version(8).download("yolov8", location="/kaggle/working/datasets/humps-bumps-potholes")
    DATASET_ROOT = dataset.location

    with open(f"{DATASET_ROOT}/data.yaml") as f:
        exported = yaml.safe_load(f)
    names = exported["names"]
    if isinstance(names, list):
        names = {i: n for i, n in enumerate(names)}
    train_split, val_split = "train/images", "valid/images"
else:
    # elangovan001/speed12 ships no data.yaml; names/order confirmed by inspecting label files (see markdown above).
    # Mount depth under /kaggle/input varies (e.g. input/speed12 vs input/datasets/.../speed12), so search recursively.
    print("under /kaggle/input:", sorted(str(p) for p in Path("/kaggle/input").glob("*")))
    hits = [p.parent.parent for p in Path("/kaggle/input").rglob("train/images") if (p.parent.parent / "test/images").is_dir()]
    assert hits, "no train/images + test/images found anywhere under /kaggle/input - check the attached dataset"
    DATASET_ROOT = str(hits[0])
    names = {0: "Pothole", 1: "Speed Bump"}

    # Carve a val split out of train/ (test/ stays untouched - see markdown above for why it's not recombined).
    train_images = sorted(
        str(p) for p in (Path(DATASET_ROOT) / "train/images").glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    rng = random.Random(42)
    rng.shuffle(train_images)
    n_val = max(1, round(0.10 * len(train_images)))
    val_images, train_only_images = train_images[:n_val], train_images[n_val:]

    Path("data").mkdir(exist_ok=True)
    Path("data/train_list.txt").write_text("\n".join(train_only_images) + "\n")
    Path("data/val_list.txt").write_text("\n".join(val_images) + "\n")
    print(f"train/: {len(train_images)} images -> {len(train_only_images)} train / {len(val_images)} val "
          f"(test/: untouched, used only for final evaluation)")

    train_split = str(Path("data/train_list.txt").resolve())
    val_split = str(Path("data/val_list.txt").resolve())

assert len(names) == 2, f"expected 2 classes (Pothole, Speed Bump), found {names}"
print("dataset root:", DATASET_ROOT, "| classes:", names)

with open("data/data.yaml", "w") as f:
    yaml.safe_dump(
        {"path": DATASET_ROOT, "train": train_split, "val": val_split, "test": "test/images", "names": names},
        f,
        sort_keys=False,
    )
print(open("data/data.yaml").read())


## 5. Configure and run training

Mirrors the 6-stage ladder documented in `train.py`. By default this runs only the final model (stage 6: GhostConv + VoVGSCSPC + LGDH + IoU/NWD loss). Add earlier stage names to `RUN_STAGES` to reproduce the full ablation study - each stage is trained independently.

`yolo26n.yaml` / `yolo26n.pt` in stage 1 are stock Ultralytics names resolved from the package itself (and auto-downloaded from the internet), not a file this notebook writes.


In [ ]:
import subprocess
import sys

EPOCHS = 150
BATCH = 16
IMGSZ = 640
DEVICE = "0"  # "0,1" on a 2x T4 session
WORKERS = 4
RUNS_DIR = "/kaggle/working/runs"

STAGES = [
    dict(cfg="yolo26n.yaml", name="s1_yolo26n"),
    dict(cfg="models/ablation/gv-yolo26n-p2.yaml", name="s2_p2"),
    dict(cfg="models/ablation/gv-yolo26n-p2-lgdh.yaml", name="s3_lgdh"),
    dict(cfg="models/ablation/gv-yolo26n-p2-lgdh-ghost.yaml", name="s4_ghost"),
    dict(cfg="models/gv-yolo26n.yaml", name="s5_vov"),
    dict(cfg="models/gv-yolo26n.yaml", name="s6_gv_yolo26", nwd=True),
]
RUN_STAGES = ["s6_gv_yolo26"]  # e.g. add "s1_yolo26n", ..., "s5_vov" to run the full ladder


def run_stage(stage):
    cmd = [
        sys.executable, "train.py",
        "--cfg", stage["cfg"],
        "--name", stage["name"],
        "--data", "data/data.yaml",
        "--epochs", str(EPOCHS),
        "--batch", str(BATCH),
        "--imgsz", str(IMGSZ),
        "--device", DEVICE,
        "--workers", str(WORKERS),
        "--project", RUNS_DIR,
    ]
    if stage.get("nwd"):
        cmd.append("--nwd")
    print("+", " ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


by_name = {s["name"]: s for s in STAGES}
for stage_name in RUN_STAGES:
    run_stage(by_name[stage_name])


## 6. Inspect results

In [ ]:
from pathlib import Path

from IPython.display import Image, display

last_name = RUN_STAGES[-1]
res_dir = Path(RUNS_DIR) / last_name
display(Image(filename=str(res_dir / "results.png")))
display(Image(filename=str(res_dir / "confusion_matrix.png")))
print("best weights:", res_dir / "weights" / "best.pt")


## 7. Validate on the test split

In [ ]:
val_cmd = [
    sys.executable, "val.py",
    "--weights", str(res_dir / "weights" / "best.pt"),
    "--data", "data/data.yaml",
    "--split", "test",
]
subprocess.run(val_cmd, cwd=PROJECT_DIR, check=True)


## 8. Save the trained weights as a notebook output

In [ ]:
import shutil

out_path = f"/kaggle/working/{last_name}_best.pt"
shutil.copy(res_dir / "weights" / "best.pt", out_path)
print("saved", out_path, "- download it from this notebook's Output tab.")
